# PS-S06E07: Pseudo-Labeling Generation

This notebook generates the pseudo-labeled training dataset (`data/train_pseudo.csv`) by ensembling the predictions of the 5 foundational models on the test set, selecting the top 10% most confident predictions, and appending them to the original training dataset.

---

## 1. The Pseudo-Labeling Process
1. **Ensemble Test Predictions**: The out-of-fold (OOF) predictions and test set predictions of the 5 base models (CatBoost, LightGBM, XGBoost, Random Forest, Neural Network Tabular ResNet) are loaded.
2. **Confidence Computation**: For each row in the test set, the blended probability vector $p = [p_{\text{at-risk}}, p_{\text{unhealthy}}, p_{\text{fit}}]$ is computed. The confidence score is defined as the maximum class probability:
   $$\text{Confidence} = \max(p)$$
3. **High-Confidence Selection**: The top 10% (29,575 rows) most confident test rows are selected and assigned their predicted class as a pseudo-label:
   $$\text{Pseudo-Label} = \text{argmax}(p)$$
4. **Append to Training Data**: The selected pseudo-labeled rows are merged with the original training set (`data/train.csv`) to produce the expanded dataset `data/train_pseudo.csv`.

---

## 2. Where the Pseudo Dataset is Destined to be Used
The generated `data/train_pseudo.csv` is loaded in the pseudo-labeled training notebooks:
- [ps-s06e07-lightgbm-pseudo.ipynb](file:///home/tarter/repos/kaggle/kaggle-playground-series/2026/s06e07/ps-s06e07-lightgbm-pseudo.ipynb)
- [ps-s06e07-xgboost-pseudo.ipynb](file:///home/tarter/repos/kaggle/kaggle-playground-series/2026/s06e07/ps-s06e07-xgboost-pseudo.ipynb)

By training on this expanded dataset, these models learn the distribution of the test set directly. The predictions and OOF files produced by these pseudo-trained models (`lgb_pseudo` and `xgb_pseudo`) are then fed into the final blending and stacking pipelines, adding significant diversity to the final ensemble.

---

## 3. How the Blending Weights Were Derived
The `blend_weights` used to ensemble the test probabilities:
- **XGBoost**: 62.67%
- **CatBoost**: 14.37%
- **LightGBM**: 14.37%
- **Random Forest**: 4.39%
- **NN Tabular ResNet**: 4.19%

These weights were derived using a **Caruana-style Hill Climbing Algorithm** (forward selection ensemble search) on the out-of-fold (OOF) cross-validation predictions of the 5 base models. 

The search was run over 500 iterations to find the exact combination of models that maximizes **Balanced Accuracy**. Because XGBoost was the strongest individual model with well-calibrated probabilities, it was selected as the starting point and received the highest weight, while CatBoost, LightGBM, and the other models were blended in to correct for residual errors.


In [1]:
import os
import glob
import numpy as np
import pandas as pd

from ps_s06e07_experiment_setup import ExperimentSetup


In [2]:
setup = ExperimentSetup(
    model_name='PseudoLabeling',
    use_gpu=False,
    perform_rfe=False,
    perform_optuna_tuning=False
)

# Target mapping
target_mapping = {'at-risk': 0, 'unhealthy': 1, 'fit': 2}
inverse_target_mapping = {v: k for k, v in target_mapping.items()}


In [3]:
# Load original train and test datasets
train_df = setup.read_dataset('training')
test_df = setup.read_dataset('test')

print(f"Original Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


TRAINING DATASET

   id health_condition  sleep_duration  heart_rate    bmi  \
0   0        unhealthy            5.22        70.6  25.66   
1   1          at-risk            5.53        71.3  25.84   
2   2        unhealthy            5.29        75.4  24.54   
3   3        unhealthy            4.70        77.2  23.13   
4   4          at-risk            7.23        73.4  28.44   

   calorie_expenditure  step_count  exercise_duration  water_intake diet_type  \
0               2174.0      1326.0               19.8          1.86       veg   
1               1966.0      9891.0               49.9          1.26   non-veg   
2               2688.0     14216.0               38.1          1.60       veg   
3               2630.0      7174.0               59.9          2.02       veg   
4               2560.0      6584.0               46.0          2.25       veg   

  stress_level sleep_quality physical_activity_level smoking_alcohol  gender  \
0         high       average               seden

In [5]:
# Local convention: predictions stored under predictions/s06e07/
# Kaggle convention: mount dataset input that contains predictions/
if setup.running_in_kaggle():
    pred_dir = '/kaggle/input/notebooks/stephentarter/ps-s06e07-*/predictions'
else:
    pred_dir = 'predictions'

EXCLUDE_PREFIXES = ('blending', 'stacking', 'lgb_pseudo', 'xgb_pseudo')

def _is_base_model_file(path):
    name = os.path.basename(path)
    return not name.startswith(EXCLUDE_PREFIXES)

test_files = sorted(f for f in glob.glob(f'{pred_dir}/*_test_probs.csv') if _is_base_model_file(f))
model_names = [os.path.basename(f).replace('_test_probs.csv', '') for f in test_files]

print("Loading test predictions for base models:")
test_probs_dict = {}
for m, filepath in zip(model_names, test_files):
    if os.path.exists(filepath):
        df = pd.read_csv(filepath).set_index('id')
        # Ensure column alignment
        cols = [f"prob_at_risk", f"prob_unhealthy", f"prob_fit"]
        if not all(c in df.columns for c in cols):
            # Check if columns are prefixed
            cols = [f"{m}_prob_at_risk", f"{m}_prob_unhealthy", f"{m}_prob_fit"]
        probs = df.reindex(test_df['id'])[cols].values
        test_probs_dict[m] = probs
        print(f"  Loaded {m} test predictions shape: {probs.shape}")
    else:
        print(f"  WARNING: {filepath} not found!")


Loading test predictions for base models:
  Loaded catboost test predictions shape: (295753, 3)
  Loaded lgb test predictions shape: (295753, 3)
  Loaded nn_tabular_resnet test predictions shape: (295753, 3)
  Loaded rf test predictions shape: (295753, 3)
  Loaded xgb test predictions shape: (295753, 3)


In [6]:
# Blend test probabilities using hill climbing weights
# (xgb: 0.6267, catboost: 0.1437, lgb: 0.1437, rf: 0.0439, nn_tabular_resnet: 0.0419)
blend_weights = {
    'xgb': 0.6267,
    'catboost': 0.1437,
    'lgb': 0.1437,
    'rf': 0.0439,
    'nn_tabular_resnet': 0.0419
}

# Normalize weights for available models
available_models = [m for m in model_names if m in test_probs_dict]
total_w = sum(blend_weights[m] for m in available_models)
norm_weights = {m: blend_weights[m] / total_w for m in available_models}

blend_probs = np.zeros((len(test_df), 3))
for m in available_models:
    blend_probs += test_probs_dict[m] * norm_weights[m]

print("Successfully blended test probabilities.")


Successfully blended test probabilities.


In [7]:
# For each row, get the maximum probability and predicted class
max_probs = np.max(blend_probs, axis=1)
pred_classes = np.argmax(blend_probs, axis=1)

# Select the top 10% most confident rows (approx 29,576 rows)
num_pseudo = int(len(test_df) * 0.10)
sorted_indices = np.argsort(-max_probs)
pseudo_indices = sorted_indices[:num_pseudo]

print(f"Selected the top 10% ({num_pseudo}) most confident test rows.")
print(f"Confidence threshold: {max_probs[pseudo_indices[-1]]:.4f}")


Selected the top 10% (29575) most confident test rows.
Confidence threshold: 0.9739


In [8]:
# Slice the selected test rows
pseudo_df = test_df.iloc[pseudo_indices].copy()

# Assign mapped string labels to the health_condition target column
pseudo_df['health_condition'] = [inverse_target_mapping[c] for c in pred_classes[pseudo_indices]]

# Concatenate with train.csv
# Match train columns: ID is preserved for indexing alignment in cross-validation
common_cols = [c for c in train_df.columns if c in pseudo_df.columns]
train_pseudo_df = pd.concat([train_df[common_cols], pseudo_df[common_cols]], ignore_index=True)

# Save to data/train_pseudo.csv
os.makedirs('data', exist_ok=True)
train_pseudo_df.to_csv('data/train_pseudo.csv', index=False)

print(f"New train_pseudo shape: {train_pseudo_df.shape}")
print("\nTarget class distribution in pseudo-labeled data:")
print(pseudo_df['health_condition'].value_counts())
print("\nTarget class distribution in final combined training dataset:")
print(train_pseudo_df['health_condition'].value_counts(normalize=True))


New train_pseudo shape: (719663, 16)

Target class distribution in pseudo-labeled data:
health_condition
unhealthy    17750
fit          11825
Name: count, dtype: int64

Target class distribution in final combined training dataset:
health_condition
at-risk      0.823387
unhealthy    0.104874
fit          0.071739
Name: proportion, dtype: float64
